In [1]:
!pip -q install rank_bm25 konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 26.0 MB/s eta 0:00:00


In [2]:
import json
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from konlpy.tag import Komoran

try:
  import chromadb
except ImportError:
  !pip install chromadb
  import chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found e

In [3]:
model_name = 'koe5'
hf_id = 'nlpai-lab/KoE5'
db_path = '/content/drive/MyDrive/프젝랩/임베딩 모델/chroma_db/koe5'

top_k = 6

In [4]:
client = chromadb.PersistentClient(path=db_path)
collection = client.get_collection("client")

In [5]:
all_data = collection.get()
ids = all_data['ids']
docs = all_data['documents']
metadatas = all_data['metadatas']

print(len(ids))

8980


In [6]:
id_to_doc = {doc_id: doc for doc_id, doc in zip(ids, docs)}
id_to_meta = {doc_id: meta for doc_id, meta in zip(ids, metadatas)}

In [7]:
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('huggingface')
login(token=HF_TOKEN)

In [8]:
embedding_model = SentenceTransformer(hf_id, device='cpu', token=HF_TOKEN)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/741 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [9]:
komoran = Komoran()

KEEP_POS = {"NNG", "NNP", "SL", "XR"}
# NNG: 일반명사, NNP: 고유명서, SL: 외국어/영어 약어, XR: 어근

EXCLUDE_TOKENS = {"TITLE", "SUMMARY"}

def tokenize(text):
  if text is None:
    return []

  tokens = []

  for word, pos in komoran.pos(str(text)):
    if pos in KEEP_POS:
      token = word.strip()

      if token in EXCLUDE_TOKENS:
        continue

      if token:
        tokens.append(token)

  return tokens

In [10]:
tokenized = []

for doc in tqdm(docs, desc="BM25 tokenizing"):
  tokenized.append(tokenize(doc))

print("sample tokens: ", tokenized[0][:30])

BM25 tokenizing:   0%|          | 0/8980 [00:00<?, ?it/s]

sample tokens:  ['부정', '수급', '위험', '사이렌', '정지원', '사업', '부정', '수급', '사례', '전파', '부정', '수급', '예방']


In [11]:
EXCLUDE_TOKENS = {"TITLE", "SUMMARY"}

tokenized = [
    [
        token for token in doc_tokens
        if token not in EXCLUDE_TOKENS
    ]
    for doc_tokens in tokenized
]

In [12]:
print(tokenized[:30])

[['부정', '수급', '위험', '사이렌', '정지원', '사업', '부정', '수급', '사례', '전파', '부정', '수급', '예방'], ['건설업', '사망', '사고', '유발', 'SIF', '철골', '작업', '건설', '현장', '사망', '사고', '유발', '요인', '이해', '인', '포', '그래픽', '반영', 'OPS', '형태', '제작'], ['위험성', '평가', '인정', '안전', '혜택', '위험성', '평가', '인정', '안전', '혜택', 'OPS'], ['사업장', '위험성', '평가', 'KRAS', '간편', '사업장', '위험성', '평가', 'KRAS', '간편', 'OPS'], ['건설업', '사망', '사고', '유발', 'SIF', '전기', '설비', '작업', '건설', '현장', '사망', '사고', '유발', '요인', '이해', '인', '포', '그래픽', '반영', 'OPS', '형태', '제작'], ['건설업', '사망', '사고', '유발', 'SIF', '거푸집', '동바리', '조립', '해체', '작업', '건설', '현장', '사망', '사고', '유발', '요인', '이해', '인', '포', '그래픽', '반영', 'OPS', '형태', '제작'], ['건설업', '사망', '사고', '유발', 'SIF', '기계', '설비', '건설', '현장', '사망', '사고', '유발', '요인', '이해', '인', '포', '그래픽', '반영', 'OPS', '형태', '제작'], ['건설업', '사망', '사고', '유발', 'SIF', '도장', '작업', '건설', '현장', '사망', '사고', '유발', '요인', '이해', '인', '포', '그래픽', '반영', 'OPS', '형태', '제작'], ['건설업', '사망', '사고', '유발', 'SIF', '철거', '해체', '작업', '건설', '현장', '사망', '사고', '유발', '요인', '이해', 

In [13]:
bm25 = BM25Okapi(tokenized)

In [14]:
def dense_retriever(query, dense_top_k):
  query_embedding = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0].tolist()

  results = collection.query(
      query_embeddings=[query_embedding],
      n_results=dense_top_k,
      include=["documents", "metadatas", "distances"]
  )

  dense_ids = results["ids"][0]
  dense_distances = results["distances"][0]

  return dense_ids, dense_distances

In [15]:
def bm25_retriever(query, bm25_top_k, min_score=0):
  query_tokens = tokenize(query)
  scores = bm25.get_scores(query_tokens)
  top_indices = np.argsort(scores)[::-1][:bm25_top_k]

  bm25_ids = [ids[i] for i in top_indices if scores[i] > min_score]
  bm25_scores = [scores[i] for i in top_indices if scores[i] > min_score]

  return bm25_ids, bm25_scores

In [16]:
def rrf_fusion(rankings, rrf_k): # rankings: [[dense_id1, dense_id2, ...], [bm25_id1, bm25_id2, ...]]
  scores = {}

  for ranking in rankings:
    for rank, doc_id in enumerate(ranking, 1):
      if doc_id not in scores:
        scores[doc_id] = 0
      scores[doc_id] += 1 / (rrf_k + rank)

  fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)

  return fused


In [17]:
def hybrid_retriever(query, dense_top_k, bm25_top_k, rrf_k, final_k):
  dense_ids,dense_distances =  dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k)
  final_results = fused[:final_k]

  output = []

  for rank, (doc_id, rrf_score) in enumerate(final_results, 1):
    meta = id_to_meta[doc_id]
    doc = id_to_doc[doc_id]

    output.append({
        "rank": rank,
        "doc_id": doc_id,
        "rrf_score": rrf_score,
        "title": meta.get("title"),
        "document": doc,
        "metadata": meta
    })

  return output

In [19]:
query = "지게차 운전자의 안전수칙이 있는 OPS 검색해줘"
top_k = 10
rrf_k = 60
final_k = 6

output = hybrid_retriever(query, top_k, top_k, rrf_k, final_k)
output

[{'rank': 1,
  'doc_id': 'list4_39817',
  'rrf_score': 0.03177805800756621,
  'title': '지게차 안전수칙이 생명수칙',
  'document': '[TITLE] 지게차 안전수칙이 생명수칙\n[SUMMARY] 지게차 안전수칙이 생명수칙 1. 면허 소지자가 운전 2. 운전자 시야 확보 3. 좌석 안전벨트 착용',
  'metadata': {'language': 'Korean',
   'record_id': 'list4_39817',
   'source_list_kind': 'other',
   'publisher': '교육미디어',
   'views': '24830',
   'detail_url': 'https://portal.kosha.or.kr/archive/cent-archive/master-arch/master-list4/master-detail4?medSeq=39817',
   'created_at': '2018-09-04',
   'title': '지게차 안전수칙이 생명수칙',
   'summary': '지게차 안전수칙이 생명수칙 1. 면허 소지자가 운전 2. 운전자 시야 확보 3. 좌석 안전벨트 착용'}},
 {'rank': 2,
  'doc_id': 'list5_41780',
  'rrf_score': 0.02919863597612958,
  'title': '(교안) 지게차 운전자의 안전보건(특수형태근로종사자_건설기계)',
  'document': '[TITLE] (교안) 지게차 운전자의 안전보건(특수형태근로종사자_건설기계)\n[SUMMARY] 사고사망 감소를 위한 안전보건 교안',
  'metadata': {'detail_url': 'https://portal.kosha.or.kr/archive/cent-archive/master-arch/master-list5/master-detail5?medSeq=41780',
   'record_id': 'list5_41780',
   'v

In [21]:
query = "밀폐공간 질식사고 예방자료 검색"
output = hybrid_retriever(query, top_k, top_k, rrf_k, final_k)
output

[{'rank': 1,
  'doc_id': 'list3_43647',
  'rrf_score': 0.0315136476426799,
  'title': '밀폐공간 질식재해예방 안전작업 가이드',
  'document': '[TITLE] 밀폐공간 질식재해예방 안전작업 가이드\n[SUMMARY] 밀폐공간 질식재해예방 안전작업 가이드 개정',
  'metadata': {'summary': '밀폐공간 질식재해예방 안전작업 가이드 개정',
   'language': 'Korean',
   'source_list_kind': 'booklet',
   'detail_url': 'https://portal.kosha.or.kr/archive/cent-archive/master-arch/master-list3/master-detail3?medSeq=43647',
   'record_id': 'list3_43647',
   'title': '밀폐공간 질식재해예방 안전작업 가이드',
   'views': '48339',
   'created_at': '2021-12-02',
   'publisher': '사업총괄본부'}},
 {'rank': 2,
  'doc_id': 'list3_46430',
  'rrf_score': 0.03131881575727918,
  'title': '`24년 밀폐공간 질식재해예방 안전작업 가이드',
  'document': '[TITLE] `24년 밀폐공간 질식재해예방 안전작업 가이드\n[SUMMARY] `24년 밀폐공간 질식재해예방 안전작업 가이드',
  'metadata': {'created_at': '2024-09-11',
   'detail_url': 'https://portal.kosha.or.kr/archive/cent-archive/master-arch/master-list3/master-detail3?medSeq=46430',
   'title': '`24년 밀폐공간 질식재해예방 안전작업 가이드',
   'publisher': '산업보

In [28]:
test_dataset_path = "/content/drive/MyDrive/프젝랩/test_dataset_수정.json"
result_dir = "/content/drive/MyDrive/프젝랩/Eval results/hybrid_retriever"

with open(test_dataset_path, 'r', encoding="utf-8") as f:
  test_dataset = json.load(f)

In [19]:
def evaluation(test_dataset, dense_top_k, bm25_top_k, rrf_k, final_k):
  rows = []

  for idx, test_data in enumerate(test_dataset, 1):
    query = test_data['query']
    gold_title = test_data['title']

    outputs = hybrid_retriever(query, dense_top_k, bm25_top_k, rrf_k, final_k)

    correct_rank = None

    row = {
        "query_idx": idx,
        "query": query,
        "gold_title": gold_title,
    }

    for output in outputs:
      rank = output['rank']
      row[f'top{rank}_title'] = output['title']
      row[f'top{rank}_rrf_score'] = output['rrf_score']

      if correct_rank is None and gold_title == output['title']:
        correct_rank = rank

    row['correct_rank'] = correct_rank
    row['top1_accuracy'] = int(correct_rank == 1)
    row['top6_recall'] = int( correct_rank is not None and correct_rank <= final_k)

    rows.append(row)

  detail_df = pd.DataFrame(rows)
  test_size = len(test_dataset)
  top1_accuracy = int(detail_df['top1_accuracy'].sum()) / test_size
  top6_recall = int(detail_df['top6_recall'].sum()) / test_size

  summary = {
      'model': 'koe5 + bm25',
      'test_size': test_size,
      'dense_top_k': dense_top_k,
      'bm25_top_k': bm25_top_k,
      'rrf_k': rrf_k,
      'final_k': final_k,
      'top1_accuracy': round(top1_accuracy * 100, 2),
      'top6_recall': round(top6_recall * 100, 2),
      'top1_hit': int(detail_df['top1_accuracy'].sum()),
      'top6_hit': int(detail_df['top6_recall'].sum())
  }

  return detail_df, summary


In [20]:
def save_result(dense_top_k, bm25_top_k, rrf_k, final_k):
  hybrid_detail_df, hybrid_summary = evaluation(test_dataset, dense_top_k, bm25_top_k, rrf_k, final_k)

  hybrid_summary_df = pd.DataFrame([hybrid_summary])

  result_dir = "/content/drive/MyDrive/프젝랩/Eval results/hybrid_retriever"
  result_dir = Path(result_dir)
  result_dir.mkdir(parents=True, exist_ok=True)

  detail_path = result_dir / f'hybrid_detail_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'
  summary_path = result_dir / f'hybrid_summary_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'

  hybrid_detail_df.to_csv(detail_path, index=False, encoding='utf-8-sig')
  hybrid_summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

  return hybrid_summary

In [29]:
dense_top_k = 10
bm25_top_k = 10
rrf_k = 60
final_k = 6

hybrid_summary = save_result(dense_top_k, bm25_top_k, rrf_k, final_k)
hybrid_summary

{'model': 'koe5 + bm25',
 'test_size': 28,
 'dense_top_k': 10,
 'bm25_top_k': 10,
 'rrf_k': 60,
 'final_k': 6,
 'top1_accuracy': 7.14,
 'top6_recall': 28.57,
 'top1_hit': 2,
 'top6_hit': 8}

In [30]:
dense_top_k = 20
bm25_top_k = 10
rrf_k = 60
final_k = 6

hybrid_summary = save_result(dense_top_k, bm25_top_k, rrf_k, final_k)
hybrid_summary

{'model': 'koe5 + bm25',
 'test_size': 28,
 'dense_top_k': 20,
 'bm25_top_k': 10,
 'rrf_k': 60,
 'final_k': 6,
 'top1_accuracy': 7.14,
 'top6_recall': 32.14,
 'top1_hit': 2,
 'top6_hit': 9}

In [ ]:
dense_top_k = 30
bm25_top_k = 10
rrf_k = 60
final_k = 6

hybrid_summary = save_result(dense_top_k, bm25_top_k, rrf_k, final_k)
hybrid_summary

{'model': 'koe5 + bm25',
 'test_size': 28,
 'dense_top_k': 30,
 'bm25_top_k': 10,
 'rrf_k': 60,
 'final_k': 6,
 'top1_accuracy': 7.14,
 'top6_recall': 28.57,
 'top1_hit': 2,
 'top6_hit': 8}

In [ ]:
dense_top_k = 30
bm25_top_k = 20
rrf_k = 60
final_k = 6

hybrid_summary = save_result(dense_top_k, bm25_top_k, rrf_k, final_k)
hybrid_summary

{'model': 'koe5 + bm25',
 'test_size': 28,
 'dense_top_k': 30,
 'bm25_top_k': 20,
 'rrf_k': 60,
 'final_k': 6,
 'top1_accuracy': 7.14,
 'top6_recall': 28.57,
 'top1_hit': 2,
 'top6_hit': 8}

reranking 추가

In [27]:
from sentence_transformers import CrossEncoder

In [28]:
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [29]:
def add_reranker_retriever(query, dense_top_k, bm25_top_k, rrf_k, candidates_k, final_k):
  dense_ids,dense_distances =  dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k)
  fused = fused[:candidates_k]

  candidate_ids = []
  candidate_rrf_scores = {}

  seen = set()

  for doc_id, rrf_score in fused:
    if doc_id not in seen:
      seen.add(doc_id)
      candidate_ids.append(doc_id)
      candidate_rrf_scores[doc_id] = rrf_score

  pairs = [[query, id_to_doc[doc_id]] for doc_id in candidate_ids]

  rerank_scores = reranker.predict(pairs)
  reranked = sorted(zip(candidate_ids, rerank_scores), key=lambda x: x[1], reverse=True)

  final_results = reranked[:final_k]

  return final_results

In [30]:
def evaluation_reranker(test_dataset, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k):
  rows = []

  for idx, test_data in enumerate(test_dataset, 1):
    query = test_data['query']
    gold_title = test_data['title']

    reranked = add_reranker_retriever(query, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)

    correct_rank = None

    row = {
        "query_idx": idx,
        "query": query,
        "gold_title": gold_title,
    }

    for rank, (doc_id, rerank_score) in enumerate(reranked, 1):
      meta = id_to_meta[doc_id]
      title = meta.get('title')

      row[f'top{rank}_doc_id'] = doc_id
      row[f'top{rank}_title'] = title
      row[f'top{rank}_rerank_score'] = float(rerank_score)

      if correct_rank is None and gold_title == title:
        correct_rank = rank

    row['correct_rank'] = correct_rank
    row['top1_accuracy'] = int(correct_rank == 1)
    row['top6_recall'] = int( correct_rank is not None and correct_rank <= final_k)

    rows.append(row)


  detail_df = pd.DataFrame(rows)
  test_size = len(test_dataset)
  top1_accuracy = int(detail_df['top1_accuracy'].sum()) / test_size
  top6_recall = int(detail_df['top6_recall'].sum()) / test_size

  summary = {
      'model': 'koe5 + bm25 + bge-reranker-v2-m3',
      'test_size': test_size,
      'dense_top_k': dense_top_k,
      'bm25_top_k': bm25_top_k,
      'rrf_k': rrf_k,
      'final_k': final_k,
      'top1_accuracy': round(top1_accuracy * 100, 2),
      'top6_recall': round(top6_recall * 100, 2),
      'top1_hit': int(detail_df['top1_accuracy'].sum()),
      'top6_hit': int(detail_df['top6_recall'].sum())
  }

  return detail_df, summary


In [31]:
def save_result(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k):
  reranker_detail_df, reranker_summary = evaluation_reranker(test_dataset, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)

  reranker_summary_df = pd.DataFrame([reranker_summary])

  result_dir = Path("/content/drive/MyDrive/프젝랩/Eval results/add_reranker_retriever")
  result_dir.mkdir(parents=True, exist_ok=True)

  detail_path = result_dir / f'reranker_detail_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'
  summary_path = result_dir / f'reranker_summary_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'

  reranker_detail_df.to_csv(detail_path, index=False, encoding='utf-8-sig')
  reranker_summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

  return reranker_summary

In [32]:
query = "자동라인 청소 중 사고 예방수칙"

dense_top_k = 30
bm25_top_k = 20
rrf_k = 60
candidate_k = 30
final_k = 6

final_results = add_reranker_retriever(query, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1646/125333510.py", line 9, in <cell line: 0>
    final_results = add_reranker_retriever(query, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1646/2363725929.py", line 21, in add_reranker_retriever
    rerank_scores = reranker.predict(pairs)
                    ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/util/decorators.py", line 41, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/us

TypeError: object of type 'NoneType' has no len()

In [ ]:
for rank, result in enumerate(final_results, 1):
  doc_id = result[0]
  meta = id_to_meta[doc_id]
  title = meta.get('title')
  print(f"{title}, {result[1]}")

[TITLE] (기타제조업)자동화설비 취급작업

[SUMMARY] 기계기구 및 작업공정과 관련된 내용을 이해하기 쉽게 OPL(One Point Lesson)로 제작하였습니다.


In [ ]:
for rank, result in enumerate(final_results, 1):
  doc_id = result[0]
  meta = id_to_meta[doc_id]
  title = meta.get('title')
  print(f"{title}, {result[1]}")

In [ ]:
dense_top_k = 30
bm25_top_k = 20
rrf_k = 60
candidate_k = 20
final_k = 6

reranker_summary = save_result(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)
reranker_summary

In [ ]:
dense_top_k = 30
bm25_top_k = 20
rrf_k = 60
candidate_k = 30
final_k = 6

reranker_summary = save_result(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)
reranker_summary

bge-reranker-base 시도

In [46]:
from sentence_transformers import CrossEncoder

In [34]:
base_reranker = CrossEncoder("BAAI/bge-reranker-base")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [35]:
def add_reranker_retriever1(query, dense_top_k, bm25_top_k, rrf_k, candidates_k, final_k):
  dense_ids,dense_distances =  dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k)
  fused = fused[:candidates_k]

  candidate_ids = []
  candidate_rrf_scores = {}

  seen = set()

  for doc_id, rrf_score in fused:
    if doc_id not in seen:
      seen.add(doc_id)
      candidate_ids.append(doc_id)
      candidate_rrf_scores[doc_id] = rrf_score

  pairs = [[query, id_to_doc[doc_id]] for doc_id in candidate_ids]

  rerank_scores = base_reranker.predict(pairs)
  reranked = sorted(zip(candidate_ids, rerank_scores), key=lambda x: x[1], reverse=True)

  final_results = reranked[:final_k]

  return final_results

In [36]:
def evaluation_reranker(test_dataset, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k):
  rows = []

  for idx, test_data in enumerate(test_dataset, 1):
    query = test_data['query']
    gold_title = test_data['title']

    reranked = add_reranker_retriever1(query, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)

    correct_rank = None

    row = {
        "query_idx": idx,
        "query": query,
        "gold_title": gold_title,
    }

    for rank, (doc_id, rerank_score) in enumerate(reranked, 1):
      meta = id_to_meta[doc_id]
      title = meta.get('title')

      row[f'top{rank}_doc_id'] = doc_id
      row[f'top{rank}_title'] = title
      row[f'top{rank}_rerank_score'] = float(rerank_score)

      if correct_rank is None and gold_title == title:
        correct_rank = rank

    row['correct_rank'] = correct_rank
    row['top1_accuracy'] = int(correct_rank == 1)
    row['top6_recall'] = int( correct_rank is not None and correct_rank <= final_k)

    rows.append(row)


  detail_df = pd.DataFrame(rows)
  test_size = len(test_dataset)
  top1_accuracy = int(detail_df['top1_accuracy'].sum()) / test_size
  top6_recall = int(detail_df['top6_recall'].sum()) / test_size

  summary = {
      'model': 'koe5 + bm25 + bge-reranker-v2-m3',
      'test_size': test_size,
      'dense_top_k': dense_top_k,
      'bm25_top_k': bm25_top_k,
      'rrf_k': rrf_k,
      'final_k': final_k,
      'top1_accuracy': round(top1_accuracy * 100, 2),
      'top6_recall': round(top6_recall * 100, 2),
      'top1_hit': int(detail_df['top1_accuracy'].sum()),
      'top6_hit': int(detail_df['top6_recall'].sum())
  }

  return detail_df, summary


In [37]:
def save_result1(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k):
  reranker_detail_df, reranker_summary = evaluation_reranker(test_dataset, dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)

  reranker_summary_df = pd.DataFrame([reranker_summary])

  result_dir = Path("/content/drive/MyDrive/프젝랩/Eval results/add_reranker_retriever1")
  result_dir.mkdir(parents=True, exist_ok=True)

  detail_path = result_dir / f'reranker1_detail_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'
  summary_path = result_dir / f'reranker1_summary_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'

  reranker_detail_df.to_csv(detail_path, index=False, encoding='utf-8-sig')
  reranker_summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

  return reranker_summary

In [38]:
dense_top_k = 30
bm25_top_k = 20
rrf_k = 60
candidate_k = 20
final_k = 6

reranker1_summary = save_result1(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)
reranker1_summary

KeyboardInterrupt: 

In [ ]:
dense_top_k = 30
bm25_top_k = 20
rrf_k = 60
candidate_k = 30
final_k = 6

reranker1_summary = save_result1(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)
reranker1_summary

In [ ]:
dense_top_k = 30
bm25_top_k = 10
rrf_k = 30
candidate_k = 30
final_k = 6

reranker1_summary = save_result1(dense_top_k, bm25_top_k, rrf_k, candidate_k, final_k)
reranker1_summary

bm25 성능 재평가

In [23]:
def find_gold_rank(doc_ids, gold_title):
  for rank, doc_id in enumerate(doc_ids, 1):
    if id_to_meta[doc_id]['title'] == gold_title:
      return rank
  return None

In [24]:
def union_recall_experiment(test_dataset, k_set = (6, 10, 30, 50, 100)):
  save_dir = Path("/content/drive/MyDrive/프젝랩/Eval results/union_recall_experiment")
  save_dir.mkdir(parents=True, exist_ok=True)

  max_k = max(k_set)
  detail_rows = []

  for idx, item in enumerate(test_dataset, 1):
    query = item['query']
    gold_title = item['title']

    dense_ids, dense_scores = dense_retriever(query, max_k)
    bm25_ids, bm25_scores = bm25_retriever(query, max_k)

    for k in k_set:
      dense_k_ids = dense_ids[:k]
      bm25_k_ids = bm25_ids[:k]

      dense_rank = find_gold_rank(dense_k_ids, gold_title)
      bm25_rank = find_gold_rank(bm25_k_ids, gold_title)

      dense_hit = dense_rank is not None
      bm25_hit = bm25_rank is not None

      union_ids = list(dict.fromkeys(dense_k_ids + bm25_k_ids)) # dict.fromkeys는 순서를 유지하면서 중복 제거
      union_rank = find_gold_rank(union_ids, gold_title)
      union_hit = union_rank is not None

      dense_set = set(dense_k_ids)
      bm25_set = set(bm25_k_ids)
      union_set = dense_set | bm25_set

      overlap_jaccard = (len(dense_set & bm25_set) / len(union_set)) if len(union_set) > 0 else 0

      detail_rows.append({
          'query_idx': idx,
          'query': query,
          'gold_title': gold_title,
          'k': k,
          'dense_hit': int(dense_hit),
          'bm25_hit': int(bm25_hit),
          'union_hit': int(union_hit),
          'dense_rank': dense_rank,
          'bm25_rank': bm25_rank,
          'union_rank': union_rank,
          'bm25_added_gold': int((not dense_hit) and bm25_hit),
          'dense_added_gold': int((not bm25_hit) and dense_hit),
          'both_hit': int(dense_hit and bm25_hit),
          'neither_hit': int((not dense_hit) and (not bm25_hit)),
          "dense_candidate_size": len(dense_k_ids),
          "bm25_candidate_size": len(bm25_k_ids),
          "union_candidate_size": len(union_ids),
          'overlap_jaccard': overlap_jaccard,
      })

  detail_df = pd.DataFrame(detail_rows)

  summary_rows = []

  for k, group in detail_df.groupby("k"):
      n = len(group)

      dense_hit_count = group["dense_hit"].sum()
      bm25_hit_count = group["bm25_hit"].sum()
      union_hit_count = group["union_hit"].sum()

      summary_rows.append({
            "k": k,
            "test_size": n,

            "dense_recall": round(dense_hit_count / n * 100, 2),
            "bm25_recall": round(bm25_hit_count / n * 100, 2),
            "union_oracle_recall": round(union_hit_count / n * 100, 2),

            "union_gain_over_dense": round((union_hit_count - dense_hit_count) / n * 100, 2),

            "dense_hit_count": int(dense_hit_count),
            "bm25_hit_count": int(bm25_hit_count),
            "union_hit_count": int(union_hit_count),

            "bm25_added_gold_count": int(group["bm25_added_gold"].sum()),
            "dense_only_gold_count": int(group["dense_added_gold"].sum()),
            "both_hit_count": int(group["both_hit"].sum()),
            "neither_hit_count": int(group["neither_hit"].sum()),

            "avg_dense_candidate_size": round(group["dense_candidate_size"].mean(), 2),
            "avg_bm25_candidate_size": round(group["bm25_candidate_size"].mean(), 2),
            "avg_union_candidate_size": round(group["union_candidate_size"].mean(), 2),
            "avg_overlap_jaccard": round(group["overlap_jaccard"].mean(), 4),
      })

  summary_df = pd.DataFrame(summary_rows).sort_values("k")

  detail_path = save_dir / "union_recall_detail.csv"
  summary_path = save_dir / "union_recall_summary.csv"

  detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
  summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

  print(f"detail saved: {detail_path}")
  print(f"summary saved: {summary_path}")

  return detail_df, summary_df

In [31]:
detail_df, summary_df = union_recall_experiment(test_dataset)
summary_df

detail saved: /content/drive/MyDrive/프젝랩/Eval results/union_recall_experiment/union_recall_detail.csv
summary saved: /content/drive/MyDrive/프젝랩/Eval results/union_recall_experiment/union_recall_summary.csv


,k,test_size,dense_recall,bm25_recall,union_oracle_recall,union_gain_over_dense,dense_hit_count,bm25_hit_count,union_hit_count,bm25_added_gold_count,dense_only_gold_count,both_hit_count,neither_hit_count,avg_dense_candidate_size,avg_bm25_candidate_size,avg_union_candidate_size,avg_overlap_jaccard
0,6,28,32.14,14.29,39.29,7.14,9,4,11,2,7,2,17,6.0,6.0,10.79,0.1278
1,10,28,35.71,28.57,39.29,3.57,10,8,11,1,3,7,17,10.0,10.0,17.21,0.1820
2,30,28,46.43,42.86,57.14,10.71,13,12,16,3,4,9,12,30.0,30.0,49.07,0.2450
3,50,28,57.14,57.14,71.43,14.29,16,16,20,4,4,12,8,50.0,50.0,83.14,0.2221
4,100,28,60.71,64.29,75.00,14.29,17,18,21,4,3,14,7,100.0,100.0,171.18,0.1818


bm25가 dense retrieval의 성능을 보완하긴 하지만, top rank 성능은 dense가 bm25보다 뛰어남. 정답을 상위에 rank하는 능력이 dense보다 떨어짐.
\
따라서 dense와의 비중을 1:1로 하지 않고 dense 가중치를 더 크게 해야함\
\
overlap_jaccard가 낮음 > dense와 bm25의 후보군 차이가 큼 > bm25가 새로운 후보를 가져오기 때문에 dense를 보완하는 방식이 유의적일 것임 하지만 정답이 아닌 후보를 가져온다면 오히려 후보군에 노이즈를 가져올 가능성이 있음




Neither hit case

k=100 기준\
24년 밀폐공간 질식재해예방 안전작업 가이드 \
지게차 재해사례와 위험요인 \
(기타제조업)자동화설비 취급작업 \
LOTO 작업절차 바로알기
[안전기준] 원동기·회전축 위험방지 등

\
\

- 24년 밀폐공간 질식재해예방 안전작업 가이드\
query: \
산소 부족으로 쓰러지는 사고 예방 OPS > 산소 부족에서 dense 검색이 되면 좋았겠지만 그러지 못했던 것 같음, 필요하다면 도메인 사전을 만들어서 밀폐와 산소공간을 연결시키면 도움될 것 같음

유해가스 측정 관련 작업자료 > 자료 representation의 부족 문제. 이것 또한 도메인 사전 구축해 유해가스와 질식을 연결시키면 도움될 것 같음

>> bm25 검색에서 자료, 예방, 가이드 등의 키워드 배제 필요성

- 지게차 재해사례와 위험요인\
query: \
포크에 화물 싣고 이동할 때 주의사항 자료 > query가 title에 비해 구체적. gold가 하나만 지정되어 있어 생긴 평가 한계 가능성.지게차와 포크를 연결시키지 못하는 retriever의 한계

상하차 작업 안전보건 OPS 찾아줘 > 위와 동일

- (기타제조업)자동화설비 취급작업\
query: 자동라인 청소 중 사고 예방수칙 > query 확장 or domain synonym 보강 필요

- LOTO 작업절차 바로알기\
query: 설비 정비 중 갑자기 작동하는 사고 관련 자료 > 자료 representation의 부족 문제

- [안전기준] 원동기·회전축 위험방지 등
query: 회전체 끼임사고 예방 OPS 검색 > query 확장 or domain synonym 보강 필요

In [32]:
def rrf_fusion(rankings, rrf_k, weights=None): # rankings: [[dense_id1, dense_id2, ...], [bm25_id1, bm25_id2, ...]]
  scores = {}

  if weights is None:
    weights = [1.0] * len(rankings)

  for ranking, weight in zip(rankings, weights):
    for rank, doc_id in enumerate(ranking, 1):
      if doc_id not in scores:
        scores[doc_id] = 0
      scores[doc_id] += weight * (1 / (rrf_k + rank))

  fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)

  return fused


In [33]:
komoran = Komoran()

KEEP_POS = {"NNG", "NNP", "SL", "XR"}
# NNG: 일반명사, NNP: 고유명서, SL: 외국어/영어 약어, XR: 어근

EXCLUDE_TOKENS = {"TITLE", "SUMMARY", "예방", "자료", "가이드", "안전", "가이드라인", "보건"}

def tokenize(text):
  if text is None:
    return []

  tokens = []

  for word, pos in komoran.pos(str(text)):
    if pos in KEEP_POS:
      token = word.strip()

      if token in EXCLUDE_TOKENS:
        continue

      if token:
        tokens.append(token)

  return tokens

In [34]:
tokenized = []

for doc in tqdm(docs, desc="BM25 tokenizing"):
  tokenized.append(tokenize(doc))

print("sample tokens: ", tokenized[0][:30])

BM25 tokenizing:   0%|          | 0/8980 [00:00<?, ?it/s]

sample tokens:  ['부정', '수급', '위험', '사이렌', '정지원', '사업', '부정', '수급', '사례', '전파', '부정', '수급']


In [35]:
bm25 = BM25Okapi(tokenized)

In [36]:
def bm25_retriever(query, bm25_top_k, min_score=0):
  query_tokens = tokenize(query)
  scores = bm25.get_scores(query_tokens)
  top_indices = np.argsort(scores)[::-1][:bm25_top_k]

  bm25_ids = [ids[i] for i in top_indices if scores[i] > min_score]
  bm25_scores = [scores[i] for i in top_indices if scores[i] > min_score]

  return bm25_ids, bm25_scores

In [37]:
def hybrid_retriever(query, dense_top_k, bm25_top_k, rrf_k, final_k):
  dense_ids,dense_distances =  dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k)
  final_results = fused[:final_k]

  output = []

  for rank, (doc_id, rrf_score) in enumerate(final_results, 1):
    meta = id_to_meta[doc_id]
    doc = id_to_doc[doc_id]

    output.append({
        "rank": rank,
        "doc_id": doc_id,
        "rrf_score": rrf_score,
        "title": meta.get("title"),
        "document": doc,
        "metadata": meta
    })

  return output

In [38]:
def union_recall_experiment(test_dataset, k_set = (6, 10, 30, 50, 100)):
  save_dir = Path("/content/drive/MyDrive/프젝랩/Eval results/union_recall_experiment")
  save_dir.mkdir(parents=True, exist_ok=True)

  max_k = max(k_set)
  detail_rows = []

  for idx, item in enumerate(test_dataset, 1):
    query = item['query']
    gold_title = item['title']

    dense_ids, dense_scores = dense_retriever(query, max_k)
    bm25_ids, bm25_scores = bm25_retriever(query, max_k)

    for k in k_set:
      dense_k_ids = dense_ids[:k]
      bm25_k_ids = bm25_ids[:k]

      dense_rank = find_gold_rank(dense_k_ids, gold_title)
      bm25_rank = find_gold_rank(bm25_k_ids, gold_title)

      dense_hit = dense_rank is not None
      bm25_hit = bm25_rank is not None

      union_ids = list(dict.fromkeys(dense_k_ids + bm25_k_ids)) # dict.fromkeys는 순서를 유지하면서 중복 제거
      union_rank = find_gold_rank(union_ids, gold_title)
      union_hit = union_rank is not None

      dense_set = set(dense_k_ids)
      bm25_set = set(bm25_k_ids)
      union_set = dense_set | bm25_set

      overlap_jaccard = (len(dense_set & bm25_set) / len(union_set)) if len(union_set) > 0 else 0

      detail_rows.append({
          'query_idx': idx,
          'query': query,
          'gold_title': gold_title,
          'k': k,
          'dense_hit': int(dense_hit),
          'bm25_hit': int(bm25_hit),
          'union_hit': int(union_hit),
          'dense_rank': dense_rank,
          'bm25_rank': bm25_rank,
          'union_rank': union_rank,
          'bm25_added_gold': int((not dense_hit) and bm25_hit),
          'dense_added_gold': int((not bm25_hit) and dense_hit),
          'both_hit': int(dense_hit and bm25_hit),
          'neither_hit': int((not dense_hit) and (not bm25_hit)),
          "dense_candidate_size": len(dense_k_ids),
          "bm25_candidate_size": len(bm25_k_ids),
          "union_candidate_size": len(union_ids),
          'overlap_jaccard': overlap_jaccard,
      })

  detail_df = pd.DataFrame(detail_rows)

  summary_rows = []

  for k, group in detail_df.groupby("k"):
      n = len(group)

      dense_hit_count = group["dense_hit"].sum()
      bm25_hit_count = group["bm25_hit"].sum()
      union_hit_count = group["union_hit"].sum()

      summary_rows.append({
            "k": k,
            "test_size": n,

            "dense_recall": round(dense_hit_count / n * 100, 2),
            "bm25_recall": round(bm25_hit_count / n * 100, 2),
            "union_oracle_recall": round(union_hit_count / n * 100, 2),

            "union_gain_over_dense": round((union_hit_count - dense_hit_count) / n * 100, 2),

            "dense_hit_count": int(dense_hit_count),
            "bm25_hit_count": int(bm25_hit_count),
            "union_hit_count": int(union_hit_count),

            "bm25_added_gold_count": int(group["bm25_added_gold"].sum()),
            "dense_only_gold_count": int(group["dense_added_gold"].sum()),
            "both_hit_count": int(group["both_hit"].sum()),
            "neither_hit_count": int(group["neither_hit"].sum()),

            "avg_dense_candidate_size": round(group["dense_candidate_size"].mean(), 2),
            "avg_bm25_candidate_size": round(group["bm25_candidate_size"].mean(), 2),
            "avg_union_candidate_size": round(group["union_candidate_size"].mean(), 2),
            "avg_overlap_jaccard": round(group["overlap_jaccard"].mean(), 4),
      })

  summary_df = pd.DataFrame(summary_rows).sort_values("k")

  detail_path = save_dir / "union_recall_detail2.csv"
  summary_path = save_dir / "union_recall_summary2.csv"

  detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
  summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

  print(f"detail saved: {detail_path}")
  print(f"summary saved: {summary_path}")

  return detail_df, summary_df

In [39]:
detail_df2, summary_df2 = union_recall_experiment(test_dataset)
summary_df2

detail saved: /content/drive/MyDrive/프젝랩/Eval results/union_recall_experiment/union_recall_detail2.csv
summary saved: /content/drive/MyDrive/프젝랩/Eval results/union_recall_experiment/union_recall_summary2.csv


,k,test_size,dense_recall,bm25_recall,union_oracle_recall,union_gain_over_dense,dense_hit_count,bm25_hit_count,union_hit_count,bm25_added_gold_count,dense_only_gold_count,both_hit_count,neither_hit_count,avg_dense_candidate_size,avg_bm25_candidate_size,avg_union_candidate_size,avg_overlap_jaccard
0,6,28,32.14,21.43,42.86,10.71,9,6,12,3,6,3,16,6.0,6.00,10.79,0.1278
1,10,28,35.71,32.14,42.86,7.14,10,9,12,2,3,7,16,10.0,10.00,17.50,0.1610
2,30,28,46.43,39.29,53.57,7.14,13,11,15,2,4,9,13,30.0,30.00,50.14,0.2189
3,50,28,57.14,60.71,71.43,14.29,16,17,20,4,3,13,8,50.0,50.00,84.36,0.2062
4,100,28,60.71,60.71,71.43,10.71,17,17,20,3,3,14,8,100.0,98.64,171.79,0.1710


bm25 corpus 개선 후 bm25 recall 개선됨. 특히 상위 rank의 후보군에 gold title을 더 많이 포함\
또한 k=50에서 bm25_recall이 dense_recall을 뛰어넘음. 후보군을 넓게 잡으면 dense가 못가져온 후보군 가져올 수 있음\
앞으로의 실험에서는 k=50으로 진행 예정\
하지만 neither hit case는 여전히 9개로 해결되지 않음

In [40]:
def hybrid_retriever(query, dense_top_k, bm25_top_k, rrf_k, weights, final_k):
  dense_ids,dense_distances =  dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k, weights)
  final_results = fused[:final_k]

  output = []

  for rank, (doc_id, rrf_score) in enumerate(final_results, 1):
    meta = id_to_meta[doc_id]
    doc = id_to_doc[doc_id]

    output.append({
        "rank": rank,
        "doc_id": doc_id,
        "rrf_score": rrf_score,
        "title": meta.get("title"),
        "document": doc,
        "metadata": meta
    })

  return output

In [41]:
def evaluation(test_dataset, dense_top_k, bm25_top_k, rrf_k, weights, final_k):
  rows = []

  for idx, test_data in enumerate(test_dataset, 1):
    query = test_data['query']
    gold_title = test_data['title']

    outputs = hybrid_retriever(query, dense_top_k, bm25_top_k, rrf_k, weights, final_k)

    correct_rank = None

    row = {
        "query_idx": idx,
        "query": query,
        "gold_title": gold_title,
    }

    for output in outputs:
      rank = output['rank']
      row[f'top{rank}_title'] = output['title']
      row[f'top{rank}_rrf_score'] = output['rrf_score']

      if correct_rank is None and gold_title == output['title']:
        correct_rank = rank

    row['correct_rank'] = correct_rank
    row['top1_accuracy'] = int(correct_rank == 1)
    row['top6_recall'] = int( correct_rank is not None and correct_rank <= final_k)

    rows.append(row)

  detail_df = pd.DataFrame(rows)
  test_size = len(test_dataset)
  top1_accuracy = int(detail_df['top1_accuracy'].sum()) / test_size
  top6_recall = int(detail_df['top6_recall'].sum()) / test_size

  summary = {
      'model': 'koe5 + bm25',
      'test_size': test_size,
      'dense_top_k': dense_top_k,
      'bm25_top_k': bm25_top_k,
      'rrf_k': rrf_k,
      'final_k': final_k,
      'top1_accuracy': round(top1_accuracy * 100, 2),
      'top6_recall': round(top6_recall * 100, 2),
      'top1_hit': int(detail_df['top1_accuracy'].sum()),
      'top6_hit': int(detail_df['top6_recall'].sum())
  }

  return detail_df, summary


In [42]:
def save_result(dense_top_k, bm25_top_k, rrf_k, weights, final_k):
  hybrid_detail_df, hybrid_summary = evaluation(test_dataset, dense_top_k, bm25_top_k, rrf_k, weights, final_k)

  hybrid_summary_df = pd.DataFrame([hybrid_summary])

  result_dir = "/content/drive/MyDrive/프젝랩/Eval results/hybrid_retriever_tuned"
  result_dir = Path(result_dir)
  result_dir.mkdir(parents=True, exist_ok=True)

  detail_path = result_dir / f'hybrid_detail_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'
  summary_path = result_dir / f'hybrid_summary_{dense_top_k}_{bm25_top_k}_{rrf_k}_{final_k}.csv'

  hybrid_detail_df.to_csv(detail_path, index=False, encoding='utf-8-sig')
  hybrid_summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

  return hybrid_summary

In [53]:
dense_top_k = 50
bm25_top_k = 50
rrf_k = 60
weights = [0.7, 0.3]
final_k = 6

summary = save_result(dense_top_k, bm25_top_k, rrf_k, weights, final_k)
summary

{'model': 'koe5 + bm25',
 'test_size': 28,
 'dense_top_k': 50,
 'bm25_top_k': 50,
 'rrf_k': 60,
 'final_k': 6,
 'top1_accuracy': 3.57,
 'top6_recall': 32.14,
 'top1_hit': 1,
 'top6_hit': 9}

In [54]:
dense_top_k = 50
bm25_top_k = 50
rrf_k = 30
weights = [0.7, 0.3]
final_k = 6

summary = save_result(dense_top_k, bm25_top_k, rrf_k, weights, final_k)
summary

{'model': 'koe5 + bm25',
 'test_size': 28,
 'dense_top_k': 50,
 'bm25_top_k': 50,
 'rrf_k': 30,
 'final_k': 6,
 'top1_accuracy': 3.57,
 'top6_recall': 32.14,
 'top1_hit': 1,
 'top6_hit': 9}

In [43]:
def run_weighted_rrf_grid(
    test_dataset,
    dense_top_k=50,
    bm25_top_k=50,
    final_k=6,
    rrf_ks=(5, 10, 20, 30, 60, 100),
    weight_sets=([0.5, 0.5], [0.6, 0.4], [0.7, 0.3], [0.8, 0.2], [0.9, 0.1])
):
    rows = []

    for rrf_k in rrf_ks:
        for weights in weight_sets:
            _, summary = evaluation(
                test_dataset=test_dataset,
                dense_top_k=dense_top_k,
                bm25_top_k=bm25_top_k,
                rrf_k=rrf_k,
                final_k=final_k,
                weights=weights
            )

            rows.append({
                "dense_top_k": dense_top_k,
                "bm25_top_k": bm25_top_k,
                "rrf_k": rrf_k,
                "dense_weight": weights[0],
                "bm25_weight": weights[1],
                "top1_accuracy": summary["top1_accuracy"],
                "top6_recall": summary["top6_recall"],
                "top1_hit": summary["top1_hit"],
                "top6_hit": summary["top6_hit"],
            })

    grid_df = pd.DataFrame(rows)
    grid_df = grid_df.sort_values(
        by=["top6_recall", "top1_accuracy"],
        ascending=False
    )

    return grid_df

In [44]:
grid_df = run_weighted_rrf_grid(
    test_dataset,
    dense_top_k=50,
    bm25_top_k=50,
    final_k=6
)

grid_df

,dense_top_k,bm25_top_k,rrf_k,dense_weight,bm25_weight,top1_accuracy,top6_recall,top1_hit,top6_hit
2,50,50,5,0.7,0.3,14.29,35.71,4,10
3,50,50,5,0.8,0.2,14.29,35.71,4,10
8,50,50,10,0.8,0.2,14.29,35.71,4,10
18,50,50,30,0.8,0.2,14.29,35.71,4,10
1,50,50,5,0.6,0.4,10.71,35.71,3,10
28,50,50,100,0.8,0.2,10.71,35.71,3,10
12,50,50,20,0.7,0.3,7.14,35.71,2,10
17,50,50,30,0.7,0.3,7.14,35.71,2,10
22,50,50,60,0.7,0.3,7.14,35.71,2,10
27,50,50,100,0.7,0.3,7.14,35.71,2,10


rrf_k가 낮은 경우 top1_accuracy가 높은 것을 확인함.\
상위 rank의 영향력이 커져 top1에 영향을 준 것으로 보임\
하이퍼 파라미터 설정\
dense_top_k = 50\
bm25_top_k = 50\
rrf_k = 10   \
weights = [0.8, 0.2]\
final_k = 6

실험 결과 최종 top6_recall을 크게 개선하지는 못함. bm25와 dense의 합집합에 포함된 gold title을 rrf가 top6안으로 끌ㅇ올리지 못한 것으로 보임. 따라서 여기에 reranking 추가

In [47]:
base_reranker = CrossEncoder("BAAI/bge-reranker-base")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [48]:
def add_base_reranker_retriever(query, dense_top_k, bm25_top_k, rrf_k, weights, candidates_k, final_k):
  dense_ids,dense_distances =  dense_retriever(query, dense_top_k)
  bm25_ids, bm25_scores = bm25_retriever(query, bm25_top_k)

  fused = rrf_fusion([dense_ids, bm25_ids], rrf_k, weights)
  fused = fused[:candidates_k]

  candidate_ids = []
  candidate_rrf_scores = {}

  seen = set()

  for doc_id, rrf_score in fused:
    if doc_id not in seen:
      seen.add(doc_id)
      candidate_ids.append(doc_id)
      candidate_rrf_scores[doc_id] = rrf_score

  pairs = [[query, id_to_doc[doc_id]] for doc_id in candidate_ids]

  rerank_scores = base_reranker.predict(pairs)
  reranked = sorted(zip(candidate_ids, rerank_scores), key=lambda x: x[1], reverse=True)

  final_results = reranked[:final_k]

  return final_results

In [49]:
def evaluation_reranker(test_dataset, dense_top_k, bm25_top_k, rrf_k, weights, candidate_k, final_k):
  rows = []

  for idx, test_data in enumerate(test_dataset, 1):
    query = test_data['query']
    gold_title = test_data['title']

    reranked = add_base_reranker_retriever(query, dense_top_k, bm25_top_k, rrf_k, weights, candidate_k, final_k)

    correct_rank = None

    row = {
        "query_idx": idx,
        "query": query,
        "gold_title": gold_title,
    }

    for rank, (doc_id, rerank_score) in enumerate(reranked, 1):
      meta = id_to_meta[doc_id]
      title = meta.get('title')

      row[f'top{rank}_doc_id'] = doc_id
      row[f'top{rank}_title'] = title
      row[f'top{rank}_rerank_score'] = float(rerank_score)

      if correct_rank is None and gold_title == title:
        correct_rank = rank

    row['correct_rank'] = correct_rank
    row['top1_accuracy'] = int(correct_rank == 1)
    row['top6_recall'] = int( correct_rank is not None and correct_rank <= final_k)

    rows.append(row)


  detail_df = pd.DataFrame(rows)
  test_size = len(test_dataset)
  top1_accuracy = int(detail_df['top1_accuracy'].sum()) / test_size
  top6_recall = int(detail_df['top6_recall'].sum()) / test_size

  summary = {
      'model': 'koe5 + bm25 + bge-reranker-base',
      'test_size': test_size,
      'dense_top_k': dense_top_k,
      'bm25_top_k': bm25_top_k,
      'rrf_k': rrf_k,
      'weights': weights,
      'candidate_k': candidate_k,
      'final_k': final_k,
      'top1_accuracy': round(top1_accuracy * 100, 2),
      'top6_recall': round(top6_recall * 100, 2),
      'top1_hit': int(detail_df['top1_accuracy'].sum()),
      'top6_hit': int(detail_df['top6_recall'].sum())
  }

  return detail_df, summary


In [50]:
def save_result(dense_top_k, bm25_top_k, rrf_k, weights, candidate_k, final_k, result_dir):
  detail_df, summary = evaluation_reranker(test_dataset, dense_top_k, bm25_top_k, rrf_k, weights, candidate_k, final_k)

  summary_df = pd.DataFrame([summary])

  result_dir = Path(result_dir)
  result_dir.mkdir(parents=True, exist_ok=True)

  detail_path = result_dir / f'detail_{dense_top_k}_{bm25_top_k}_{rrf_k}_{weights}_{candidate_k}_{final_k}.csv'
  summary_path = result_dir / f'summary_{dense_top_k}_{bm25_top_k}_{rrf_k}_{weights}_{candidate_k}_{final_k}.csv'

  detail_df.to_csv(detail_path, index=False, encoding='utf-8-sig')
  summary_df.to_csv(summary_path, index=False, encoding='utf-8-sig')

  return summary

In [51]:
dense_top_k = 50
bm25_top_k = 50
rrf_k = 10
weights = [0.8, 0.2]
candidate_k = 50
final_k = 6

result_dir = "/content/drive/MyDrive/프젝랩/Eval results/hybrid_retriever_tuned"
summary = save_result(dense_top_k, bm25_top_k, rrf_k, weights, candidate_k, final_k, result_dir)
summary

{'model': 'koe5 + bm25 + bge-reranker-base',
 'test_size': 28,
 'dense_top_k': 50,
 'bm25_top_k': 50,
 'rrf_k': 10,
 'weights': [0.8, 0.2],
 'candidate_k': 50,
 'final_k': 6,
 'top1_accuracy': 0.0,
 'top6_recall': 28.57,
 'top1_hit': 0,
 'top6_hit': 8}

In [52]:
import pandas as pd
from pathlib import Path

dense_top_k = 50
bm25_top_k = 50
rrf_k = 10
weights = [0.8, 0.2]
final_k = 6

result_dir = "/content/drive/MyDrive/프젝랩/Eval results/hybrid_reranker_candidate_grid"

candidate_ks = [6, 10, 20]

rows = []

for candidate_k in candidate_ks:
    summary = save_result(
        dense_top_k,
        bm25_top_k,
        rrf_k,
        weights,
        candidate_k,
        final_k,
        result_dir
    )
    rows.append(summary)

candidate_grid_df = pd.DataFrame(rows)

candidate_grid_df = candidate_grid_df.sort_values(
    by=["top6_recall", "top1_accuracy"],
    ascending=False
)

candidate_grid_df

,model,test_size,dense_top_k,bm25_top_k,rrf_k,weights,candidate_k,final_k,top1_accuracy,top6_recall,top1_hit,top6_hit
0,koe5 + bm25 + bge-reranker-base,28,50,50,10,"[0.8, 0.2]",6,6,7.14,35.71,2,10
1,koe5 + bm25 + bge-reranker-base,28,50,50,10,"[0.8, 0.2]",10,6,7.14,35.71,2,10
2,koe5 + bm25 + bge-reranker-base,28,50,50,10,"[0.8, 0.2]",20,6,3.57,28.57,1,8


In [53]:
dense_top_k = 50
bm25_top_k = 50
rrf_k = 10
weights = [0.8, 0.2]
candidate_k = 30
final_k = 6

result_dir = "/content/drive/MyDrive/프젝랩/Eval results/hybrid_retriever_tuned"
summary = save_result(dense_top_k, bm25_top_k, rrf_k, weights, candidate_k, final_k, result_dir)
summary

{'model': 'koe5 + bm25 + bge-reranker-base',
 'test_size': 28,
 'dense_top_k': 50,
 'bm25_top_k': 50,
 'rrf_k': 10,
 'weights': [0.8, 0.2],
 'candidate_k': 30,
 'final_k': 6,
 'top1_accuracy': 3.57,
 'top6_recall': 28.57,
 'top1_hit': 1,
 'top6_hit': 8}